# Brain Tumour MRI — Full Training & Result Logging
**Final code set · 2026-10-22**

Run this notebook **top to bottom** in Google Colab (`Runtime > Change runtime type > GPU`).

**It is built so it cannot lose your work:**
- Everything (models, metrics, logs) is saved to **Google Drive**, so it survives disconnects.
- Deep models **checkpoint after every epoch** — if Colab crashes, just re-run the cell and it **resumes** from the last epoch, it does not start over.
- Each model runs in its **own cell**; if one fails, the others are already saved.
- **Finished stages are skipped** automatically on a re-run.
- All results are saved as **data, not images**: `results_master.csv`, per-model `classification_report.json`, `confusion_matrix.csv`, and per-epoch `history.csv`.

After it finishes, open the `metrics/results_master.csv` on your Drive — that is your results table for the final report.

In [ ]:
# 1. Mount Google Drive (results are saved here so they survive disconnects)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Configuration  --  EDIT THE TWO PATHS BELOW IF NEEDED
import os

DATA_DIR   = '/content/drive/MyDrive/MRI_DATASET'                    # must contain Training/ and Testing/
OUTPUT_DIR = '/content/drive/MyDrive/brain_tumor_results_2026-10-22' # all results are written here

# Hyperparameters (kept identical to the project so results are comparable)
EPOCHS_CNN       = 20
EPOCHS_TRANSFER  = 10
EPOCHS_EXTRACTOR = 10
BATCH    = 32
SIZE_STD = 256      # standalone CNN / InceptionV3 / Xception
SIZE_EXT = 128      # feature extractors that feed classical ML
SEED     = 42
CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']
N_CLASSES   = 4

MODELS_DIR  = os.path.join(OUTPUT_DIR, 'models')
METRICS_DIR = os.path.join(OUTPUT_DIR, 'metrics')
HISTORY_DIR = os.path.join(OUTPUT_DIR, 'history')
CONF_DIR    = os.path.join(OUTPUT_DIR, 'confusion')
LOG_DIR     = os.path.join(OUTPUT_DIR, 'logs')
for d in [MODELS_DIR, METRICS_DIR, HISTORY_DIR, CONF_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)
MASTER_CSV  = os.path.join(METRICS_DIR, 'results_master.csv')
RUN_LOG     = os.path.join(LOG_DIR, 'run_log.txt')
PROGRESS    = os.path.join(OUTPUT_DIR, 'progress.json')

# quick sanity check of the dataset location
for split in ['Training', 'Testing']:
    pth = os.path.join(DATA_DIR, split)
    print(('OK  ' if os.path.isdir(pth) else 'MISSING  '), pth)
print('Results ->', OUTPUT_DIR)

In [ ]:
# 3. Imports, seeds, logging and progress tracking
import json, time, csv, random, datetime, traceback
import numpy as np
random.seed(SEED); np.random.seed(SEED)
import tensorflow as tf
tf.random.set_seed(SEED)
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.callbacks import Callback, CSVLogger, ModelCheckpoint
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)
import joblib

print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'NONE  (Runtime > Change runtime type > GPU)')

def log(msg):
    line = '[{:%Y-%m-%d %H:%M:%S}] {}'.format(datetime.datetime.now(), msg)
    print(line)
    with open(RUN_LOG, 'a') as f:
        f.write(line + '\n')

def _progress():
    return json.load(open(PROGRESS)) if os.path.exists(PROGRESS) else {}

def mark(stage, status, info=''):
    data = _progress()
    data[stage] = {'status': status, 'info': str(info)[:300], 'time': str(datetime.datetime.now())}
    json.dump(data, open(PROGRESS, 'w'), indent=2)

def is_done(stage):
    return _progress().get(stage, {}).get('status') == 'done'

def stage(name, fn):
    # Runs fn() guarded: logs any error and keeps going so one failure never loses the rest.
    if is_done(name):
        log('SKIP ' + name + ' (already done)'); return
    mark(name, 'running')
    try:
        fn(); mark(name, 'done'); log('DONE ' + name)
    except Exception as e:
        mark(name, 'error', e)
        log('ERROR in ' + name + ': ' + repr(e) + '\n' + traceback.format_exc())
        print('\n>>> ' + name + ' failed but was isolated. Fix and re-run THIS cell; other stages are safe.')

In [ ]:
# 4. Metric logging  --  saves accuracy and every other number as DATA (csv/json), not images
def save_confusion(name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(N_CLASSES)))
    with open(os.path.join(CONF_DIR, name + '_confusion_matrix.csv'), 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['actual_vs_predicted'] + CLASS_NAMES)
        for i, row in enumerate(cm):
            w.writerow([CLASS_NAMES[i]] + [int(v) for v in row])
    return cm

def log_metrics(name, y_true, y_pred, train_time=None, note=''):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _  = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    pc, rc, fc, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(N_CLASSES)), zero_division=0)
    rep = classification_report(y_true, y_pred, target_names=CLASS_NAMES,
                                zero_division=0, output_dict=True)
    json.dump(rep, open(os.path.join(METRICS_DIR, name + '_classification_report.json'), 'w'), indent=2)
    save_confusion(name, y_true, y_pred)
    header = (['model','accuracy','precision_macro','recall_macro','f1_macro']
              + ['f1_' + c for c in CLASS_NAMES]
              + ['n_test','train_time_sec','timestamp','note'])
    row = ([name, round(acc,4), round(p,4), round(r,4), round(f,4)]
           + [round(float(x),4) for x in fc]
           + [int(len(y_true)), round(float(train_time or 0),1),
              str(datetime.datetime.now()), note])
    new = not os.path.exists(MASTER_CSV)
    with open(MASTER_CSV, 'a', newline='') as fcsv:
        w = csv.writer(fcsv)
        if new: w.writerow(header)
        w.writerow(row)
    log('{}: accuracy={:.2f}%   macro-F1={:.3f}'.format(name, acc*100, f))
    return acc

In [ ]:
# 5. Resumable training  --  checkpoints every epoch; a crash resumes, it does not restart
class EpochTracker(Callback):
    def __init__(self, path): super().__init__(); self.path = path
    def on_epoch_end(self, epoch, logs=None):
        json.dump({'epoch': int(epoch) + 1}, open(self.path, 'w'))

def resumable_fit(name, build_fn, train_ds, val_ds, epochs):
    final_path = os.path.join(MODELS_DIR, name + '.h5')
    ckpt_path  = os.path.join(MODELS_DIR, name + '_ckpt.keras')
    epoch_path = os.path.join(MODELS_DIR, name + '_epoch.json')
    if os.path.exists(final_path):
        log(name + ': final model already exists, loading')
        return load_model(final_path)
    init_epoch = 0
    if os.path.exists(ckpt_path) and os.path.exists(epoch_path):
        model = load_model(ckpt_path)
        init_epoch = json.load(open(epoch_path)).get('epoch', 0)
        log('{}: resuming from epoch {}'.format(name, init_epoch))
    else:
        model = build_fn()
    cbs = [ModelCheckpoint(ckpt_path, save_freq='epoch', save_best_only=False),
           CSVLogger(os.path.join(HISTORY_DIR, name + '_history.csv'), append=True),
           EpochTracker(epoch_path)]
    if init_epoch < epochs:
        model.fit(train_ds, validation_data=val_ds, epochs=epochs,
                  initial_epoch=init_epoch, callbacks=cbs, verbose=1)
    model.save(final_path)                       # permanent copy on Drive
    for p in (ckpt_path, epoch_path):
        try: os.remove(p)
        except OSError: pass
    return model

In [ ]:
# 6. Data loaders
def tf_datasets(size):
    common = dict(image_size=(size, size), batch_size=BATCH, label_mode='int', seed=SEED)
    tr = tf.keras.utils.image_dataset_from_directory(
        os.path.join(DATA_DIR, 'Training'), validation_split=0.1, subset='training',
        shuffle=True, **common)
    va = tf.keras.utils.image_dataset_from_directory(
        os.path.join(DATA_DIR, 'Training'), validation_split=0.1, subset='validation',
        shuffle=True, **common)
    te = tf.keras.utils.image_dataset_from_directory(
        os.path.join(DATA_DIR, 'Testing'), shuffle=False,
        image_size=(size, size), batch_size=BATCH, label_mode='int')
    norm = lambda x, y: (x / 255.0, y)
    AT = tf.data.AUTOTUNE
    return tr.map(norm).prefetch(AT), va.map(norm).prefetch(AT), te.map(norm).prefetch(AT)

def load_numpy(split, size, grayscale=False):
    import cv2
    X, y = [], []
    base = os.path.join(DATA_DIR, split)
    flag = cv2.IMREAD_GRAYSCALE if grayscale else cv2.IMREAD_COLOR
    for idx, c in enumerate(CLASS_NAMES):
        folder = os.path.join(base, c)
        for fn in sorted(os.listdir(folder)):
            img = cv2.imread(os.path.join(folder, fn), flag)
            if img is None: continue
            img = cv2.resize(img, (size, size))
            X.append(img); y.append(idx)
    X = np.asarray(X, dtype='float32') / 255.0
    if not grayscale:
        X = X[..., ::-1]           # BGR(cv2) -> RGB to match the Keras loaders
    return X, np.asarray(y)

In [ ]:
# 7. Model builders (same architectures as the project)
def build_cnn(size, feat_name='dense'):
    m = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=(size, size, 3)),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu'), layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu'), layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu'), layers.MaxPooling2D((2,2)),
        layers.Flatten(),
        layers.Dense(256, activation='relu', name=feat_name),
        layers.Dropout(0.5),
        layers.Dense(N_CLASSES, activation='softmax'),
    ])
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def _base(kind, size):
    App = tf.keras.applications.InceptionV3 if kind == 'inception' else tf.keras.applications.Xception
    b = App(input_shape=(size, size, 3), weights='imagenet', include_top=False)
    b.trainable = False
    return b

def build_transfer_standalone(kind, size=256):
    base = _base(kind, size)
    x = layers.GlobalAveragePooling2D()(base.output)
    out = layers.Dense(N_CLASSES, activation='softmax', name='dense')(x)
    m = Model(base.input, out)
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def build_transfer_extractor(kind, feat_name, size=128):
    base = _base(kind, size)
    x = layers.Flatten()(base.output)
    x = layers.Dense(256, activation='relu', name=feat_name)(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(N_CLASSES, activation='softmax')(x)
    m = Model(base.input, out)
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

In [ ]:
# 8. Prediction / feature-extraction / classical helpers, and the stage functions
def predict_ds(model, ds):
    yt, yp = [], []
    for X, y in ds:
        p = model.predict(X, verbose=0)
        yp.extend(np.argmax(p, axis=1)); yt.extend(y.numpy())
    return np.asarray(yt), np.asarray(yp)

def extract_features(model, layer_name, X, batch=64):
    ex = Model(model.input, model.get_layer(layer_name).output)
    return ex.predict(X, batch_size=batch, verbose=1)

def train_classical(tag, Xtr, ytr, Xte, yte, include_lr=False):
    # Each classifier is trained and SAVED separately, so a crash in one keeps the others.
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.svm import SVC
    from sklearn.linear_model import LogisticRegression
    specs = {'rf':  RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
             'dt':  DecisionTreeClassifier(random_state=SEED),
             'svm': SVC(probability=True, random_state=SEED, cache_size=1000)}
    if include_lr:
        specs['lr'] = LogisticRegression(max_iter=1000, n_jobs=-1)
    fitted = {}
    for key, clf in specs.items():
        mp = os.path.join(MODELS_DIR, tag + '_' + key + '.pkl')
        if os.path.exists(mp):
            fitted[key] = joblib.load(mp); log(tag + '_' + key + ': loaded (skip)'); continue
        t = time.time(); log(tag + '_' + key + ': training ...')
        clf.fit(Xtr, ytr); joblib.dump(clf, mp)
        log_metrics(tag + '_' + key, yte, clf.predict(Xte), time.time() - t)
        fitted[key] = clf
    vote_keys = [k for k in ['rf','dt','svm'] if k in fitted]
    probs = sum(fitted[k].predict_proba(Xte) for k in vote_keys) / len(vote_keys)
    log_metrics(tag + '_softvote_ensemble', yte, np.argmax(probs, axis=1))
    return fitted

# ---------- deep standalone models ----------
def run_cnn():
    tr, va, te = tf_datasets(SIZE_STD)
    m = resumable_fit('cnn_model', lambda: build_cnn(SIZE_STD, 'dense'), tr, va, EPOCHS_CNN)
    yt, yp = predict_ds(m, te); log_metrics('CNN_standalone', yt, yp)

def run_inception():
    tr, va, te = tf_datasets(SIZE_STD)
    m = resumable_fit('inceptionv3_model', lambda: build_transfer_standalone('inception'),
                      tr, va, EPOCHS_TRANSFER)
    yt, yp = predict_ds(m, te); log_metrics('InceptionV3_standalone', yt, yp)

def run_xception():
    tr, va, te = tf_datasets(SIZE_STD)
    m = resumable_fit('xception_model', lambda: build_transfer_standalone('xception'),
                      tr, va, EPOCHS_TRANSFER)
    yt, yp = predict_ds(m, te); log_metrics('Xception_standalone', yt, yp)

# ---------- deep feature extractor + classical ensemble ----------
def _run_feature_ensemble(ext_name, build_ext, layer, tag):
    tr, va, _ = tf_datasets(SIZE_EXT)
    ext = resumable_fit(ext_name, build_ext, tr, va, EPOCHS_EXTRACTOR)
    Xtr, ytr = load_numpy('Training', SIZE_EXT)
    Xte, yte = load_numpy('Testing',  SIZE_EXT)
    ftr = extract_features(ext, layer, Xtr)
    fte = extract_features(ext, layer, Xte)
    del Xtr, Xte
    train_classical(tag, ftr, ytr, fte, yte)

def run_cnn_ensemble():
    _run_feature_ensemble('cnn_ext', lambda: build_cnn(SIZE_EXT, 'dense'), 'dense', 'cnn_feat')

def run_inception_ensemble():
    _run_feature_ensemble('inceptionv3_ext',
        lambda: build_transfer_extractor('inception', 'dense_2'), 'dense_2', 'inceptionv3_feat')

def run_xception_ensemble():
    _run_feature_ensemble('xception_ext',
        lambda: build_transfer_extractor('xception', 'dense_3'), 'dense_3', 'xception_feat')

# ---------- classical ML on raw pixels ----------
def run_traditional_raw():
    Xtr, ytr = load_numpy('Training', SIZE_STD, grayscale=True)
    Xte, yte = load_numpy('Testing',  SIZE_STD, grayscale=True)
    Xtr = Xtr.reshape(len(Xtr), -1); Xte = Xte.reshape(len(Xte), -1)
    train_classical('raw', Xtr, ytr, Xte, yte, include_lr=True)

# ---------- final 6-model ensemble ----------
def run_final_ensemble():
    X256, yte = load_numpy('Testing', SIZE_STD)
    X128, _   = load_numpy('Testing', SIZE_EXT)
    probs = []
    for fn in ['cnn_model.h5', 'inceptionv3_model.h5', 'xception_model.h5']:
        mp = os.path.join(MODELS_DIR, fn)
        if os.path.exists(mp):
            probs.append(load_model(mp).predict(X256, verbose=0)); log('final: used ' + fn)
    for ext_fn, layer, tag in [('cnn_ext.h5','dense','cnn_feat'),
                               ('inceptionv3_ext.h5','dense_2','inceptionv3_feat'),
                               ('xception_ext.h5','dense_3','xception_feat')]:
        mp = os.path.join(MODELS_DIR, ext_fn)
        if not os.path.exists(mp): continue
        feats = extract_features(load_model(mp), layer, X128)
        clfs = [joblib.load(os.path.join(MODELS_DIR, tag + '_' + k + '.pkl'))
                for k in ['rf','dt','svm']
                if os.path.exists(os.path.join(MODELS_DIR, tag + '_' + k + '.pkl'))]
        if clfs:
            probs.append(sum(c.predict_proba(feats) for c in clfs) / len(clfs))
            log('final: used ' + ext_fn + ' + ' + tag)
    if not probs:
        log('final: no models found yet'); return
    P = np.stack(probs)                                  # (n_models, N, 4)
    log_metrics('FINAL_confidence_weighted', yte, np.argmax(P.sum(axis=0), axis=1),
                note=str(P.shape[0]) + ' models')
    votes = np.argmax(P, axis=2)                          # (n_models, N)
    maj = np.array([np.bincount(votes[:, i], minlength=N_CLASSES).argmax()
                    for i in range(votes.shape[1])])
    log_metrics('FINAL_majority_vote', yte, maj, note=str(P.shape[0]) + ' models')

print('Stage functions ready.')

## Run the stages below one at a time
Run each cell and wait for it to finish. If a cell crashes or Colab disconnects, just **re-run that same cell** — it resumes from the last epoch (deep models) or skips what is already saved (classical models). Already-finished stages print `SKIP`.

In [ ]:
stage('CNN_standalone',        run_cnn)

In [ ]:
stage('InceptionV3_standalone', run_inception)

In [ ]:
stage('Xception_standalone',    run_xception)

In [ ]:
stage('CNN_feature_ensemble',       run_cnn_ensemble)

In [ ]:
stage('InceptionV3_feature_ensemble', run_inception_ensemble)

In [ ]:
stage('Xception_feature_ensemble',    run_xception_ensemble)

In [ ]:
# Classical ML on raw pixels. The SVM here is the slowest step (large feature vectors).
# It is saved the moment it finishes, and skipped on any re-run.
stage('Traditional_raw_pixels', run_traditional_raw)

In [ ]:
# Run this only after the stages above have finished.
stage('FINAL_ensemble', run_final_ensemble)

## Results summary

In [ ]:
# Show the master results table and package everything for download
import pandas as pd
if os.path.exists(MASTER_CSV):
    df = pd.read_csv(MASTER_CSV).drop_duplicates('model', keep='last')
    pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
    print(df[['model','accuracy','precision_macro','recall_macro','f1_macro','train_time_sec']].to_string(index=False))
else:
    print('No results yet.')
print('\nProgress:'); print(json.dumps(_progress(), indent=2))

# Zip the whole results folder so you can download it in one click
import shutil
zip_path = '/content/brain_tumor_results_2026-10-22'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print('\nEverything is also on your Drive at:', OUTPUT_DIR)
print('Zip for download:', zip_path + '.zip')
try:
    from google.colab import files; files.download(zip_path + '.zip')
except Exception as e:
    print('(download skipped:', e, ')')

## Optional — download the dataset
Skip this if your dataset is already on Drive at `DATA_DIR` with `Training/` and `Testing/` folders.

To pull it from Kaggle, upload your `kaggle.json` (Kaggle > Account > Create New API Token) when prompted, then run:
```python
from google.colab import files; files.upload()          # choose kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip -q install kaggle
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p /content
!unzip -q /content/brain-tumor-mri-dataset.zip -d /content/drive/MyDrive/MRI_DATASET
```
After it unzips, set `DATA_DIR = '/content/drive/MyDrive/MRI_DATASET'` in cell 2 and re-run from the top.

## Troubleshooting
- **"MISSING" next to a path in cell 2** — fix `DATA_DIR`; it must contain `Training/` and `Testing/`, each with the four class folders.
- **Session crashed / disconnected** — reconnect, run cells 1–8, then re-run the stage cell that was interrupted. It resumes.
- **Out of memory on the final ensemble** — run it on its own after restarting the runtime (`Runtime > Restart`), then run cells 1–8 and the final cell.
- **Where are my numbers?** — `MyDrive/brain_tumor_results_2026-10-22/metrics/results_master.csv` (one row per model), plus `confusion/` and `history/`.